# 01 – Keşifsel Veri Analizi (EDA)

**Proje:** EUR/USD 1H Forex Zaman Serisi Tahmini  
**Ders:** Yapay Zekaya Giriş – Dönem Projesi (Hafta 3 Teslimi)  
**Öğrenci:** Bilal – `b200101025`

## Amaç

Bu notebook, ders kılavuzunun **Faz 3 (Keşifsel Veri Analizi)** adımının canlı teslimidir. İçerik:

1. Veri yüklenmesi ve tip kontrolü
2. Eksik değer / duplicate / aralıksız saat analizi
3. Tanımlayıcı istatistikler
4. Fiyat & log-getiri zaman serisi görselleştirmesi
5. Dağılım, otokorelasyon (ACF/PACF) ve volatilite analizleri
6. Seans / saat / haftanın günü etkileri
7. EDA bulgularının özeti → modelleme aşamasına taşınacak notlar

> Bu defter bir **iskelettir**. Hücreler dolduruldukça Git commit'leri ile ilerlenecektir.

## 0. Ortam ve Sabitler

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Proje kökünü sys.path'e ekle ki src/ importları çalışsın
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_raw_eurusd  # noqa: E402

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 6)

RAW_CSV = PROJECT_ROOT / 'data' / 'raw' / 'eurusd_h1.csv'
print('Python:', sys.version.split()[0])
print('CSV   :', RAW_CSV, '— exists:', RAW_CSV.exists())

## 1. Veriyi Yükle

In [ ]:
df = load_raw_eurusd(RAW_CSV)
print('shape:', df.shape)
print('time range:', df.index.min(), '->', df.index.max())
df.head()

In [ ]:
df.info()

## 2. Veri Kalitesi

- Eksik değerler
- Tekrarlayan satırlar / index
- Aralıksız olması gereken 1H seride kopukluklar (hafta sonları hariç)

In [ ]:
print('Missing values:\n', df.isna().sum(), sep='')
print('\nDuplicate index rows:', df.index.duplicated().sum())
print('Duplicate full rows  :', df.duplicated().sum())

In [ ]:
# Haftaiçi (Pazartesi-Cuma) boşluklar: 1H aralığında olması beklenen ama olmayan mumlar
full_idx = pd.date_range(df.index.min(), df.index.max(), freq='1h')
missing = full_idx.difference(df.index)
weekday_missing = missing[missing.dayofweek < 5]
print(f'Toplam beklenen saat : {len(full_idx):,}')
print(f'Veride bulunan       : {len(df):,}')
print(f'Eksik saat           : {len(missing):,}')
print(f'Hafta içi eksik saat : {len(weekday_missing):,}  (piyasa tatilleri + broker molaları)')

## 3. Tanımlayıcı İstatistikler

In [ ]:
df.describe().T

## 4. Fiyat ve Log-Getiri Görselleştirmesi

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
df['close'].plot(ax=ax[0], color='steelblue')
ax[0].set_title('EUR/USD 1H – Close')
ax[0].set_ylabel('Fiyat')

log_ret = np.log(df['close']).diff()
log_ret.plot(ax=ax[1], color='crimson', linewidth=0.4)
ax[1].set_title('Log-Getiri')
ax[1].set_ylabel('log(P_t / P_{t-1})')
plt.tight_layout();

## 5. Dağılım ve Otokorelasyon

- Log-getiri dağılımının normalliği (çarpıklık/kurtosis, QQ-plot)
- Fiyat serisinin ACF (yüksek bağımlılık bekleriz – trend)
- Log-getiri serisinin ACF (düşük; volatilite kümelenmesi `|r|` ACF'sinde görünür)

In [ ]:
# TODO: statsmodels.graphics.tsaplots ile ACF/PACF
# TODO: log-getiri QQ-plot (scipy.stats.probplot)
# TODO: rolling volatility (20H, 168H=1 hafta, 720H=1 ay)
pass

## 6. Takvim / Seans Etkileri

- Saatlik ortalama volatilite
- Gün içi seans (Asya 00-08 UTC, Avrupa 07-16 UTC, ABD 13-21 UTC)
- Haftanın günlerine göre getiri dağılımı

In [ ]:
# TODO: hour / dayofweek bazında boxplot
pass

## 7. EDA Özeti (Hoca Teslimi İçin)

Bu bölüm, notebook'un son hâlinde **doldurulacak** özet raporudur:

- Veri seti boyutu, dönemi, kalitesi ne?
- Eksik/aykırı değerler nasıl ele alındı?
- Fiyat ve getiri serisinin karakteri: trend, volatilite kümelenmesi, durağanlık
- Seans etkileri gözlemlendi mi?
- Modellemeye girerken çıkarılan dersler ve seçilen özniteliklerin gerekçesi.